# Create Holberg Prize + Nils Klim Prize Awards (PRIZE PATTERN, dual scheme)

Both the Holberg Prize and its sister Nils Klim Prize are administered
by **Universitetet i Bergen** on behalf of the Norwegian Ministry of
Education and Research. We map both to the single OpenAlex funder
**F4320323078** and distinguish them via `funder_scheme`.

Data scraped from the awarding body's own site (`holbergprisen.no` —
WordPress REST API, public, no auth, CC BY-style content), per the
runbook's prize-pattern source-authority rule. No Wikipedia/Wikidata.

**Awarding body:** Universitetet i Bergen — F4320323078

**Schema notes (prize pattern, monetary, dual scheme):**
- One row per (prize × year × laureate). No shared-year apportioning —
  Holberg/Nils Klim are awarded to a single laureate each year.
- Monetary prizes, NOK currency, **prize-specific amounts** (constant
  per prize, no year boundary):
  - Holberg Prize:     NOK 6,000,000 (~USD 550K)
  - Nils Klim Prize:   NOK 500,000   (~USD 45K)
- `lead_investigator` = the laureate (given/family name populated;
  `affiliation.name` NULL — the WP REST API doesn't expose structured
  affiliation data; following the Fields Medal precedent for NULL).
- `funder_award_id` is pre-computed by the source script as
  `{prize_slug}-{year}-{laureate_wp_slug}` (e.g.
  `holbergprisen-2026-lyndal-roper`).
- `declined` boolean column populated (always False — no declined
  Holberg/Nils Klim laureates on record); CASE present for schema
  parity with other prize-pattern notebooks.

**Prerequisites:**
- Run `scripts/local/holberg_to_s3.py` first to fetch from
  `holbergprisen.no/wp-json/wp/v2/bc_prisvinner` and upload parquet to S3.

**Data source:** https://holbergprisen.no/wp-json/wp/v2/bc_prisvinner
**S3 location:** `s3a://openalex-ingest/awards/holberg/holberg_laureates.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.holberg_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/holberg/holberg_laureates.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.holberg_raw;

## Step 1.5: Inspect raw + amount/currency note

Both prizes **are monetary** — the §6.7 amount-coverage check applies
in full (not waived like Fields Medal). Amount and currency are
pre-computed by the source script per `prize_slug`:
- `holbergprisen` → 6,000,000 NOK
- `nils-klim-prisen` → 500,000 NOK

These constants come from the official prize pages on `holbergprisen.no`
and are hardcoded in `scripts/local/holberg_to_s3.py` (PRIZE_META dict).

Source columns from the WP REST API (100% coverage on year/name/prize;
citation extracted from `content.rendered` HTML — expected ~95%
coverage since older laureates have shorter HTML bios):
`funder_award_id`, `prize_slug`, `prize_name`, `year`,
`laureate_full_name`, `laureate_given_name`, `laureate_family_name`,
`nationality`, `affiliation_country_raw`, `description`, `amount`,
`currency`, `landing_page_url`, `wp_post_id`, `declined`. All shipped
as STRING per the runbook §1.2.5 rule; TRY_CAST below as needed.

No runbook-style money-field discovery scan needed — the parquet has
named `amount`/`currency` columns by construction in the script.

In [ ]:
%sql
DESCRIBE openalex.awards.holberg_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.holberg_raw LIMIT 5;

In [ ]:
%sql
-- Coverage check on key fields, split by prize
SELECT
    prize_slug,
    COUNT(*) AS total_rows,
    COUNT(year) AS has_year,
    COUNT(laureate_full_name) AS has_name,
    COUNT(description) AS has_citation,
    COUNT(amount) AS has_amount,
    MIN(year) AS min_year,
    MAX(year) AS max_year,
    COUNT(DISTINCT year) AS distinct_years
FROM openalex.awards.holberg_raw
GROUP BY prize_slug
ORDER BY prize_slug;

## Step 1.6: Fail-fast — verify Universitetet i Bergen funder row exists

The transform in Step 2 does `CROSS JOIN openalex.common.funder WHERE
funder_id = 4320323078`. If that row isn't present, the CROSS JOIN
silently emits zero rows and the insert in Step 3 looks like it
succeeded. Assert presence here before transforming.

In [ ]:
%sql
-- Must return exactly 1 row with display_name containing 'Bergen'
-- (Universitetet i Bergen). If 0 rows, STOP — funder missing from
-- openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320323078;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.holberg_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320323078  -- Universitetet i Bergen
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT(n.prize_name, ' ', TRY_CAST(n.year AS STRING), ' \u2014 ', n.laureate_full_name) AS display_name,
    CASE
        WHEN TRY_CAST(n.declined AS BOOLEAN) = TRUE AND n.description IS NOT NULL
            THEN CONCAT('Declined the prize. ', n.description)
        WHEN TRY_CAST(n.declined AS BOOLEAN) = TRUE
            THEN 'Declined the prize.'
        ELSE n.description
    END AS description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    n.prize_name AS funder_scheme,
    'holberg_wp_rest' AS provenance,
    -- Holberg + Nils Klim ceremonies are in June in Bergen; canonical date.
    TRY_TO_DATE(CONCAT(TRY_CAST(n.year AS STRING), '-06-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(TRY_CAST(n.year AS STRING), '-06-01'), 'yyyy-MM-dd') AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    TRY_CAST(n.year AS INT) AS end_year,
    struct(
        n.laureate_given_name AS given_name,
        n.laureate_family_name AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            CAST(NULL AS STRING) AS name,
            n.nationality AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.holberg_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.year IS NOT NULL
  AND n.prize_slug IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 78

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'holberg_wp_rest' AND priority = 78;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    78 AS priority  -- Holberg priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.holberg_awards;

## Step 6: Verification

Both Holberg and Nils Klim are monetary, so §6.7 amount-coverage
check applies in full (no waiver). Expected: ~46 rows total
(~23 Holberg + ~23 Nils Klim), pct_amount = 100%, currency = NOK,
amount values exactly two: 6,000,000 (Holberg) and 500,000 (Nils Klim).

In [ ]:
%sql
SELECT COUNT(*) AS total_holberg_award_rows FROM openalex.awards.holberg_awards;

In [ ]:
%sql
-- §6.2 Schema validation on the transformed table
DESCRIBE openalex.awards.holberg_awards;

In [ ]:
%sql
-- §6.3 Data completeness (runbook canonical query)
-- NOTE: for Holberg/Nils Klim, expected pct_description ~95% — the WP REST
-- API's content.rendered HTML has a citation paragraph for most laureates
-- (regex extracts the 'tildeles ... for {citation}' sentence); a handful
-- of older laureates have shorter HTML bios and parse to NULL.
-- pct_amount expected 100% (both prizes monetary with hardcoded amounts).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_dates,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.holberg_awards;

In [ ]:
%sql
-- §6.7 amount/currency fail-fast check (NOT waived — both monetary)
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(amount), 0) AS avg_amt
FROM openalex.awards.holberg_awards;

In [ ]:
%sql
-- Sample inspection
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme, start_year, amount, currency, funder_award_id,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       lead_investigator.affiliation.country AS nationality
FROM openalex.awards.holberg_awards
ORDER BY start_year DESC, funder_scheme, family
LIMIT 12;

In [ ]:
%sql
-- Year + scheme distribution (annual since 2004 for both; ~23 of each expected)
SELECT start_year, funder_scheme, COUNT(*) AS laureates
FROM openalex.awards.holberg_awards
GROUP BY start_year, funder_scheme
ORDER BY start_year, funder_scheme;

In [ ]:
%sql
-- Scheme totals — expect ~23 of each
SELECT funder_scheme, COUNT(*) AS laureates,
       MIN(amount) AS min_amt, MAX(amount) AS max_amt
FROM openalex.awards.holberg_awards
GROUP BY funder_scheme
ORDER BY funder_scheme;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.holberg_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;